In [5]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

#client.sample_mflix #온점 표기법 -> 온점 포함 // 숫자로 시작 x 
db = client["sample_mflix"]

users = db.users
comments = db.comments 
movies = db.movies

print("ok", db.name, "collections:", [c for c in db.list_collection_names() if c in {"users","comments","movies"}])

ok sample_mflix collections: ['movies', 'users', 'comments']


In [13]:
loyal_users_pipeline = [
    {
        "$lookup": {
            "from":"comments",
            "localField": "email",
            "foreignField": "email",
            "as": "c"
        }
    },
    {
        "$addFields": {
            "commentsCount":{"$size":"$c"},
            "avgTextLen": {
                "$avg":{
                    "$map":{
                        "input":"$c",
                        "as":"x",
                        "in":{"$strLenCP":{"$ifNull":["$$x.text",""]}}
                    }
                }
            },
            "lastCommentDate":{"$max": "$c.date"}
        }
    },
    {"$match":{"commentsCount":{"$gte":10}}},
    {"$sort":{"commentsCount":-1, "avgTextLen":-1,"lastCommentDate":-1}},
    {
        "$project":{
            "_id":0,
            "name":1,
            "email":1,
            "commentsCount":1,
            "avgTextLen":1,
            "lastCommentDate":1
        }
    }
]

loyal_users =  list(users.aggregate(loyal_users_pipeline))
loyal_users[0:5]

# SQL & NoSQL -> 현업 실무 개발자들은 JOIN -> 서로 다른 데이터 테이블 혹은 컬렉션을 하나로 연결해서 가져와 처리하려면 해당 데이터의 양이 많아짐
# 방대해진 데이터를 처리하기 위한 하드웨어에 많은 무리가 가고 -> 성능&사양 제약 

[{'name': 'Mace Tyrell',
  'email': 'roger_ashton-griffiths@gameofthron.es',
  'commentsCount': 331,
  'avgTextLen': 152.49244712990938,
  'lastCommentDate': datetime.datetime(2017, 8, 2, 1, 37, 26)},
 {'name': 'Missandei',
  'email': 'nathalie_emmanuel@gameofthron.es',
  'commentsCount': 327,
  'avgTextLen': 153.0948012232416,
  'lastCommentDate': datetime.datetime(2017, 9, 11, 16, 52, 51)},
 {'name': 'The High Sparrow',
  'email': 'jonathan_pryce@gameofthron.es',
  'commentsCount': 315,
  'avgTextLen': 152.68253968253967,
  'lastCommentDate': datetime.datetime(2017, 7, 10, 4, 58, 23)},
 {'name': 'Sansa Stark',
  'email': 'sophie_turner@gameofthron.es',
  'commentsCount': 308,
  'avgTextLen': 153.42532467532467,
  'lastCommentDate': datetime.datetime(2017, 5, 2, 16, 29, 50)},
 {'name': 'Rodrik Cassel',
  'email': 'ron_donachie@gameofthron.es',
  'commentsCount': 305,
  'avgTextLen': 151.75737704918032,
  'lastCommentDate': datetime.datetime(2017, 9, 2, 6, 1, 19)}]

In [18]:
movies_report_Pipeline = [
    {
        "$facet":{
            "latest5":[
                {"$sort": {"year":-1}},
                {"$limit":5},
                {"$project":{"_id":0, "title":1,"year":1}}
            ],
            "highRatedCount":[
                {"$match":{"imdb.rating":{"$gte":8}}},
                {"$count":"count"}
            ],
             "genresTop10":[
                 {"$unwind":"$genres"},
                 {"$group":{"_id":"$genres","count":{"$sum":1}}},
                 {"$sort":{"count":-1}},
                 {"$limit":10}
             ],
            "yearlyAvgRecent10":[
                {"$group":{"_id":"$year","avgRating":{"$avg":"$imdb.rating"}}},
                {"$sort":{"_id":-1}},
                {"$limit":10},
                {
                    "$project":{
                        "_id": 0,
                        "year": "_id",
                        "avgRating":1
                        
                    }
                }

            ]
        }
    }
]

movie_report = list(movies.aggregate(movies_report_Pipeline))
movie_report

[{'latest5': [{'title': 'The Saboteurs', 'year': '2015è'},
   {'title': 'Halo: Nightfall', 'year': '2014è'},
   {'title': 'The Roosevelts: An Intimate History', 'year': '2014è'},
   {'title': 'The Weight of the Nation', 'year': '2012è'},
   {'title': 'Hit & Miss', 'year': '2012è'}],
  'highRatedCount': [{'count': 1596}],
  'genresTop10': [{'_id': 'Drama', 'count': 13789},
   {'_id': 'Comedy', 'count': 7024},
   {'_id': 'Romance', 'count': 3665},
   {'_id': 'Crime', 'count': 2678},
   {'_id': 'Thriller', 'count': 2658},
   {'_id': 'Action', 'count': 2539},
   {'_id': 'Documentary', 'count': 2129},
   {'_id': 'Adventure', 'count': 2045},
   {'_id': 'Horror', 'count': 1703},
   {'_id': 'Biography', 'count': 1404}],
  'yearlyAvgRecent10': [{'avgRating': 8.4, 'year': '_id'},
   {'avgRating': 7.25, 'year': '_id'},
   {'avgRating': 7.5, 'year': '_id'},
   {'avgRating': 7.75, 'year': '_id'},
   {'avgRating': 8.2, 'year': '_id'},
   {'avgRating': 8.3, 'year': '_id'},
   {'avgRating': 8.53333333